# Evaluación RAG — Fase 2: Sentimiento Financiero

Este notebook evalúa si el LLM (la parte generativa del RAG) interpreta correctamente el tono financiero de cada noticia (**Bullish / Bearish / Sideways**), comparado contra el ground truth real del corpus (`regimen_mercado`).

A diferencia del notebook de la Fase 1, **este no necesita Pinecone ni un modelo de embeddings** — solo compara dos columnas de datos. Puedes ejecutarlo en Colab o localmente con Python + pandas + scikit-learn.

Métricas implementadas:
- **Accuracy**
- **Precision / Recall / F1 por clase**, más **Macro-F1** y **Weighted-F1**
- **Matriz de Confusión (3×3)**
- **Cohen's Kappa Cuadrático Ponderado (QWK)** — penaliza más los errores extremos (Bearish↔Bullish) que los leves (Sideways↔cualquiera)
- **Matriz de Confusión Ponderada por Coste Financiero**

### ⚠️ Sobre las dos entradas que necesita este notebook

1. **`sentiment_ground_truth.csv`** — ya generado a partir de tu `noticias_enriquecido.json` real, con un `id` estable por fila (0 a 1390) para evitar ambigüedades: **46 URLs de tu corpus se repiten** para tickers distintos, y en 12 casos el `regimen_mercado` incluso difiere entre esas repeticiones (misma noticia, sentimiento distinto según el activo). Por eso la unión con tus predicciones se hace por `id`, nunca por `url`.
2. **Tu archivo de predicciones del LLM** — debes generarlo y subirlo tú. Formato mínimo requerido (CSV o JSON):

| Columna | Descripción |
|---|---|
| `id` | El mismo `id` de `sentiment_ground_truth.csv` (0 a 1390) — recorre ese archivo fila a fila y guarda ahí la predicción de tu RAG para esa noticia+ticker concretos. |
| `sentimiento_predicho` | Bullish / Bearish / Sideways (o sus equivalentes en español: alcista / bajista / lateral — el notebook normaliza ambos vocabularios). |

No hace falta que predigas las 1391 filas si tu pipeline aún no llega a tanto — el notebook evalúa solo sobre las filas que encuentre en tu archivo de predicciones y avisa de cuántas quedaron fuera.

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q pandas scikit-learn matplotlib seaborn

## 2. Subir archivos

Sube:
- `sentiment_ground_truth.csv` (adjunto en esta conversación)
- Tu archivo de predicciones (`.csv` o `.json`, con columnas `id` y `sentimiento_predicho`)

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Archivos subidos:", list(uploaded.keys()))

## 3. Configuración

In [ ]:
# @title Configuración
GROUND_TRUTH_PATH = "sentiment_ground_truth.csv"  # @param {type:"string"}
PREDICTIONS_PATH = "predicciones_sentimiento_sonnet.json"  # @param {type:"string"}  # nombre exacto del archivo que subiste
PREDICTIONS_FORMAT = "json"  # @param ["csv", "json"]

assert PREDICTIONS_PATH, "Indica el nombre del archivo de predicciones que subiste."


## 4. Normalización de etiquetas

El LLM puede responder en español o inglés, con mayúsculas/minúsculas distintas, o con sinónimos razonables. Esta función mapea cualquier variante común al esquema canónico `bearish` / `bullish` / `sideways` (el mismo usado en `regimen_mercado`).

In [ ]:
import re

LABEL_MAP = {
    "bearish": "bearish", "bajista": "bearish", "bajo": "bearish", "negativo": "bearish",
    "bear": "bearish", "down": "bearish", "venta": "bearish", "vender": "bearish",
    "bullish": "bullish", "alcista": "bullish", "alto": "bullish", "positivo": "bullish",
    "bull": "bullish", "up": "bullish", "compra": "bullish", "comprar": "bullish",
    "sideways": "sideways", "lateral": "sideways", "neutral": "sideways", "neutro": "sideways",
    "estable": "sideways", "flat": "sideways", "sin cambios": "sideways",
}

def normalize_label(raw):
    if raw is None:
        return None
    key = str(raw).strip().lower()
    key = re.sub(r"[^a-záéíóúñ ]", "", key)
    return LABEL_MAP.get(key, None)  # None si no reconoce la etiqueta -> se reporta como error de formato

## 5. Cargar y unir ground truth + predicciones

In [ ]:
import pandas as pd

gt = pd.read_csv(GROUND_TRUTH_PATH)

if PREDICTIONS_FORMAT == "csv":
    preds = pd.read_csv(PREDICTIONS_PATH)
else:
    preds = pd.read_json(PREDICTIONS_PATH)

assert "id" in preds.columns, "El archivo de predicciones debe tener una columna 'id'."
assert "sentimiento_predicho" in preds.columns, "El archivo de predicciones debe tener una columna 'sentimiento_predicho'."

preds["sentimiento_predicho_norm"] = preds["sentimiento_predicho"].apply(normalize_label)

sin_reconocer = preds[preds["sentimiento_predicho_norm"].isna()]
if len(sin_reconocer) > 0:
    print(f"⚠️ {len(sin_reconocer)} predicciones con etiqueta no reconocida (revisa el vocabulario):")
    print(sin_reconocer[["id", "sentimiento_predicho"]].drop_duplicates("sentimiento_predicho").to_string(index=False))

df = gt.merge(preds[["id", "sentimiento_predicho_norm"]], on="id", how="inner")
df = df.dropna(subset=["sentimiento_predicho_norm"])

print(f"Noticias en ground truth: {len(gt)}")
print(f"Predicciones recibidas: {len(preds)}")
print(f"Filas evaluables (join + etiqueta válida): {len(df)}")
if len(df) < len(gt):
    print(f"⚠️ Quedan {len(gt) - len(df)} noticias del ground truth sin predicción válida — no se incluyen en las métricas.")

df.head()

## 6. Accuracy, Precision/Recall/F1 por clase, Macro-F1 y Weighted-F1

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

CLASSES = ["bearish", "sideways", "bullish"]  # orden ordinal, usado también en el QWK más abajo

y_true = df["regimen_mercado"]
y_pred = df["sentimiento_predicho_norm"]

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

report = classification_report(y_true, y_pred, labels=CLASSES, digits=4, zero_division=0)
print(report)

macro_f1 = f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=CLASSES, average="weighted", zero_division=0)
print(f"Macro-F1:    {macro_f1:.4f}")
print(f"Weighted-F1: {weighted_f1:.4f}")

## 7. Matriz de Confusión (3×3)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
cm_df = pd.DataFrame(cm, index=[f"Real: {c}" for c in CLASSES], columns=[f"Predicho: {c}" for c in CLASSES])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Matriz de Confusión — Sentimiento Financiero")
plt.tight_layout()
plt.savefig("matriz_confusion_sentimiento.png", dpi=150)
plt.show()

cm_df

## 8. Cohen's Kappa Cuadrático Ponderado (QWK)

Como Bearish–Sideways–Bullish es una escala ordinal, el QWK penaliza cuadráticamente según la distancia del error: confundir Bearish↔Bullish (distancia 2) pesa 4 veces más que confundir Sideways con cualquiera de las otras dos (distancia 1).

In [ ]:
from sklearn.metrics import cohen_kappa_score

ORDINAL_MAP = {"bearish": 0, "sideways": 1, "bullish": 2}
y_true_ord = y_true.map(ORDINAL_MAP)
y_pred_ord = y_pred.map(ORDINAL_MAP)

qwk = cohen_kappa_score(y_true_ord, y_pred_ord, weights="quadratic")
print(f"Quadratic Weighted Kappa: {qwk:.4f}")
print("(-1 a 1; 0 = concuerda igual que el azar; 1 = concuerda perfectamente; negativo = peor que el azar)")

## 9. Matriz de Confusión Ponderada por Coste Financiero

Complementa la matriz de conteos con una matriz de "coste" fijo: confundir Sideways con cualquiera cuesta poco, pero confundir Bearish con Bullish (o viceversa) es el error catastrófico.

In [ ]:
import numpy as np

# Filas/columnas en el mismo orden que CLASSES: [bearish, sideways, bullish]
COST_MATRIX = np.array([
    [0, 1, 5],   # Real Bearish  -> Predicho [Bearish, Sideways, Bullish]
    [1, 0, 1],   # Real Sideways -> Predicho [Bearish, Sideways, Bullish]
    [5, 1, 0],   # Real Bullish  -> Predicho [Bearish, Sideways, Bullish]
])

cost_applied = cm * COST_MATRIX
mean_cost = cost_applied.sum() / cm.sum()

print("Matriz de coste aplicada (conteo × peso):")
print(pd.DataFrame(cost_applied, index=[f"Real: {c}" for c in CLASSES], columns=[f"Predicho: {c}" for c in CLASSES]))
print(f"\nCoste Medio de Confusión: {mean_cost:.4f}  (0 = perfecto; más alto = peor, con errores extremos pesando 5x)")

## 10. Casos peor clasificados (para depurar el prompt del LLM)

Muestra ejemplos donde el error fue "catastrófico" (Bearish↔Bullish, coste=5), los más urgentes de revisar.

In [ ]:
df_eval = df.copy()
df_eval["coste_error"] = [
    COST_MATRIX[ORDINAL_MAP[real]][ORDINAL_MAP[pred]]
    for real, pred in zip(y_true, y_pred)
]

peores = df_eval[df_eval["coste_error"] == 5][["id", "titulo", "ticker", "periodo", "regimen_mercado", "sentimiento_predicho_norm"]]
print(f"Errores catastróficos (Bearish↔Bullish): {len(peores)} de {len(df_eval)} ({100*len(peores)/len(df_eval):.1f}%)")
peores.head(20)

## 11. Guardar resultados

In [ ]:
summary = pd.DataFrame({
    "métrica": ["accuracy", "macro_f1", "weighted_f1", "qwk", "coste_medio_confusion", "n_evaluado"],
    "valor": [accuracy, macro_f1, weighted_f1, qwk, mean_cost, len(df_eval)],
})
summary.to_csv("resultados_sentimiento_resumen.csv", index=False)
df_eval.to_csv("resultados_sentimiento_detalle.csv", index=False)

from google.colab import files as colab_files
colab_files.download("resultados_sentimiento_resumen.csv")
colab_files.download("resultados_sentimiento_detalle.csv")
colab_files.download("matriz_confusion_sentimiento.png")

summary